# Training Delivery Delay Predictor

## Imports

In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import classification_report, confusion_matrix

import xgboost as xgb

import matplotlib.pyplot as plt
import seaborn as sns

import joblib

## Loding Data

In [2]:
data_path= '../data/processed/delivery_training_data.csv'
df= pd.read_csv(filepath_or_buffer= data_path)

In [3]:
df.head()

,order_id,estimated_days_to_deliver,actual_days_to_deliver,is_delayed,total_freight_value,total_weight_g,total_volume_cm3,is_interstate
0,00010242fe8c5a6d1ba2dd792cb16214,16,7,0,13.29,650.0,3528.0,1
1,00018f77f2f0320c557190d7a144bdd3,19,16,0,19.93,30000.0,60000.0,0
2,000229ec398224ef6ca0657da4fc703e,22,8,0,17.87,3050.0,14157.0,0
3,00024acbcdf0a6daa1e931b038114c75,12,6,0,12.79,200.0,2400.0,0
4,00042b26cf59d7ce69dfabb4e55b4fd9,41,25,0,18.14,3750.0,42000.0,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96986 entries, 0 to 96985
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   order_id                   96986 non-null  object 
 1   estimated_days_to_deliver  96986 non-null  int64  
 2   actual_days_to_deliver     96986 non-null  int64  
 3   is_delayed                 96986 non-null  int64  
 4   total_freight_value        96986 non-null  float64
 5   total_weight_g             96970 non-null  float64
 6   total_volume_cm3           96970 non-null  float64
 7   is_interstate              96986 non-null  int64  
dtypes: float64(3), int64(4), object(1)
memory usage: 5.9+ MB


In [5]:
df.describe()

,estimated_days_to_deliver,actual_days_to_deliver,is_delayed,total_freight_value,total_weight_g,total_volume_cm3,is_interstate
count,96986.000000,96986.000000,96986.000000,96986.000000,96970.000000,9.697000e+04,96986.000000
mean,24.391005,12.477543,0.080733,22.664569,2374.167227,1.723971e+04,0.640958
std,8.754967,9.540919,0.272427,21.435324,4753.040835,3.003597e+04,0.479722
min,3.000000,0.000000,0.000000,0.000000,0.000000,1.680000e+02,0.000000
25%,19.000000,7.000000,0.000000,13.790000,300.000000,2.944000e+03,0.000000
50%,24.000000,10.000000,0.000000,17.140000,750.000000,7.200000e+03,1.000000
75%,29.000000,16.000000,0.000000,23.900000,2050.000000,1.980000e+04,1.000000
max,156.000000,210.000000,1.000000,1794.960000,184400.000000,1.476000e+06,1.000000


In [7]:
df.shape

(96986, 8)

In [8]:
df.columns

Index(['order_id', 'estimated_days_to_deliver', 'actual_days_to_deliver',
       'is_delayed', 'total_freight_value', 'total_weight_g',
       'total_volume_cm3', 'is_interstate'],
      dtype='object')

In [9]:
df.dtypes

order_id                      object
estimated_days_to_deliver      int64
actual_days_to_deliver         int64
is_delayed                     int64
total_freight_value          float64
total_weight_g               float64
total_volume_cm3             float64
is_interstate                  int64
dtype: object

## Train Test Split

In data, we have actual_days_to_deliver. If we give that to the model, the model will be 100% accurate because it will just look at how long it took and compare it to the estimate. In the real world, at the moment a customer clicks "Buy," we don't know the actual delivery time. We must drop it.

In [10]:
# Dropping order_id and actual_days_to_deliver
df= df.drop(columns= ['order_id', 'actual_days_to_deliver'])

In [11]:
df.head()

,estimated_days_to_deliver,is_delayed,total_freight_value,total_weight_g,total_volume_cm3,is_interstate
0,16,0,13.29,650.0,3528.0,1
1,19,0,19.93,30000.0,60000.0,0
2,22,0,17.87,3050.0,14157.0,0
3,12,0,12.79,200.0,2400.0,0
4,41,0,18.14,3750.0,42000.0,1


In [12]:
# Separating Features and Target Variable:
X = df.drop('is_delayed', axis= 1)
y= df['is_delayed']

In [13]:
X.head()

,estimated_days_to_deliver,total_freight_value,total_weight_g,total_volume_cm3,is_interstate
0,16,13.29,650.0,3528.0,1
1,19,19.93,30000.0,60000.0,0
2,22,17.87,3050.0,14157.0,0
3,12,12.79,200.0,2400.0,0
4,41,18.14,3750.0,42000.0,1


In [14]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: is_delayed, dtype: int64

In [15]:
# Train Test Split:
X_train, X_test, y_train, y_test= train_test_split(X, y, test_size= 0.2, random_state= 42)

In [16]:
print(f'Training Features Shape: {X_train.shape}')
print(f'Testing Features Shape: {X_test.shape}')

Training Features Shape: (77588, 5)
Testing Features Shape: (19398, 5)


## Data Pre-Processing Pipeline

In [17]:
X_train.head()

,estimated_days_to_deliver,total_freight_value,total_weight_g,total_volume_cm3,is_interstate
72686,26,13.16,314.0,2772.0,0
14966,9,7.39,350.0,816.0,0
54119,13,8.29,350.0,3780.0,0
61611,29,27.21,3600.0,21600.0,1
28377,14,7.78,200.0,4800.0,0
